In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import openvino as ov
import librosa
import numpy as np
from transformers import AutoTokenizer
from IPython.display import Audio
import torch

/opt/conda/envs/sparktts/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from openvino_export.bicodec import BiCodecSemanticTokenizer, BiCodecGlobalTokenizer, BiCodecDetokenizer, BiCodecSemanticDetokenizer, BiCodecGlobalDetokenizer, BiCodecPreNet, BiCodecVocoder
from sparktts.models.bicodec import BiCodec
from openvino_export.wav2vec2 import Wav2Vec2Wrapper
from openvino_export.mel_spectrogram import MelSpectrogram
from sparktts.utils.file import load_config
from sparktts.utils.audio import load_audio

In [3]:
bicodec_config = load_config("./pretrained_models/Spark-TTS-0.5B/BiCodec/config.yaml")["audio_tokenizer"]
bicodec_config["mel_params"]

{'sample_rate': 16000, 'n_fft': 1024, 'win_length': 640, 'hop_length': 320, 'mel_fmin': 10, 'mel_fmax': None, 'num_mels': 128}

In [4]:
wav2vec = Wav2Vec2Wrapper("./pretrained_models/Spark-TTS-0.5B/wav2vec2-large-xlsr-53")
bicodec = BiCodec.load_from_checkpoint("./pretrained_models/Spark-TTS-0.5B/BiCodec")
mel_spectrogram = MelSpectrogram(bicodec_config["mel_params"])

/opt/conda/envs/sparktts/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Missing tensor: mel_transformer.spectrogram.window
Missing tensor: mel_transformer.mel_scale.fb


In [37]:
semantic_tokenizer = BiCodecSemanticTokenizer(bicodec)
global_tokenizer = BiCodecGlobalTokenizer(bicodec)

semantic_detokenizer = BiCodecSemanticDetokenizer(bicodec)
global_detokenizer = BiCodecGlobalDetokenizer(bicodec)
prenet = BiCodecPreNet(bicodec)
vocoder = BiCodecVocoder(bicodec)

semantic_tokenizer.eval()
global_tokenizer.eval()
semantic_detokenizer.eval()
global_detokenizer.eval()
prenet.eval()
vocoder.eval()

detokenizer = BiCodecDetokenizer(bicodec)
detokenizer.eval()

BiCodecDetokenizer(
  (quantizer): FactorizedVectorQuantize(
    (in_project): Conv1d(1024, 8, kernel_size=(1,), stride=(1,))
    (out_project): Conv1d(8, 1024, kernel_size=(1,), stride=(1,))
    (codebook): Embedding(8192, 8)
  )
  (speaker_encoder): SpeakerEncoder(
    (speaker_encoder): ECAPA_TDNN(
      (layer1): Conv1dReluBn(
        (conv): Conv1d(128, 512, kernel_size=(5,), stride=(1,), padding=(2,))
        (bn): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (layer2): SE_Res2Block(
        (se_res2block): Sequential(
          (0): Conv1dReluBn(
            (conv): Conv1d(512, 512, kernel_size=(1,), stride=(1,))
            (bn): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (1): Res2Conv1dReluBn(
            (convs): ModuleList(
              (0-6): 7 x Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
            )
            (bns): ModuleList(
 

In [6]:
audio, sr = librosa.load("./example/prompt_audio.wav", sr=16000)
audio.shape, sr

((159253,), 16000)

In [7]:
feat_input = torch.tensor(audio).unsqueeze(0)
feat = wav2vec(feat_input)
feat.shape, feat.max(), feat.min()

(torch.Size([1, 497, 1024]),
 tensor(739.2125, grad_fn=<MaxBackward1>),
 tensor(-302.6834, grad_fn=<MinBackward1>))

In [8]:
semantic_tokens = semantic_tokenizer(feat)
semantic_tokens.shape, semantic_tokens.max(), semantic_tokens.min()

(torch.Size([1, 497]), tensor(8161), tensor(58))

In [9]:
mel_input = torch.tensor(audio).unsqueeze(0).unsqueeze(0)
mel = mel_spectrogram(mel_input)
mel.shape, mel.max(), mel.min()

(torch.Size([1, 128, 499]), tensor(2.2982), tensor(9.5148e-07))

In [10]:
global_tokens = global_tokenizer(mel)
global_tokens.shape, global_tokens.max(), global_tokens.min()

(torch.Size([1, 1, 32]),
 tensor(4034, dtype=torch.int32),
 tensor(34, dtype=torch.int32))

In [11]:
z_q = semantic_detokenizer(semantic_tokens)
z_q.shape, z_q.max(), z_q.min()

(torch.Size([1, 1024, 497]),
 tensor(14.6566, grad_fn=<MaxBackward1>),
 tensor(-14.1637, grad_fn=<MinBackward1>))

In [12]:
d_vector = global_detokenizer(global_tokens)
d_vector.shape, d_vector.max(), d_vector.min()

(torch.Size([1, 1024]),
 tensor(1.0604, grad_fn=<MaxBackward1>),
 tensor(-0.8507, grad_fn=<MinBackward1>))

In [13]:
prenet_out = prenet(z_q, d_vector)
prenet_out.shape, prenet_out.max(), prenet_out.min()

(torch.Size([1, 1024, 497]),
 tensor(5.0460, grad_fn=<MaxBackward1>),
 tensor(-8.4986, grad_fn=<MinBackward1>))

In [14]:
wav = vocoder(prenet_out, d_vector)
wav.shape

torch.Size([1, 1, 159040])

In [15]:
Audio(audio, rate=sr)  # Play the audio to verify it loaded correctly

In [16]:
cpu_wav = wav.detach().cpu().numpy().squeeze().squeeze()
Audio(cpu_wav, rate=sr)

In [39]:
wav2 = detokenizer(semantic_tokens, global_tokens)
wav2.shape, wav2.max(), wav2.min()

(torch.Size([1, 1, 159040]),
 tensor(0.0918, grad_fn=<MaxBackward1>),
 tensor(-0.0864, grad_fn=<MinBackward1>))

In [40]:
cpu_wav2 = wav2.detach().cpu().numpy().squeeze().squeeze()
Audio(cpu_wav2, rate=sr)  # Play the detokenized audio to

In [17]:
ov_mel_spectrogram = ov.convert_model(mel_spectrogram, example_input=mel_input)

/app/openvino_export/mel_spectrogram.py:79: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if wav_with_channel.ndim != 3 or wav_with_channel.shape[1] != 1:


In [18]:
ov_wav2vec = ov.convert_model(wav2vec, example_input=feat_input)

/opt/conda/envs/sparktts/lib/python3.12/site-packages/transformers/modeling_utils.py:5006: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
/opt/conda/envs/sparktts/lib/python3.12/site-packages/transformers/modeling_utils.py:5006: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(
/opt/conda/envs/sparktts/lib/python3.12/site-packages/transformers/models/wav2vec2/modeling_wav2vec2.py:872: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  i

In [19]:
ov_semantic_tokenizer = ov.convert_model(semantic_tokenizer, example_input=feat)

/opt/conda/envs/sparktts/lib/python3.12/site-packages/torch/jit/_trace.py:166: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at aten/src/ATen/core/TensorBody.h:489.)
  if a.grad is not None:


In [20]:
ov_global_tokenizer = ov.convert_model(global_tokenizer, example_input=mel)

/app/sparktts/modules/fsq/residual_fsq.py:250: TracerWarning: Iterating over a tensor might cause the trace to be incorrect. Passing a tensor of different shape won't change the number of iterations executed (and might lead to errors or silently give incorrect results).
  zip(self.layers, self.scales)
/app/sparktts/modules/fsq/finite_scalar_quantization.py:201: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  z.shape[-1] == self.dim
/app/sparktts/modules/fsq/finite_scalar_quantization.py:154: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert zhat.shape

In [21]:
ov_semantic_detokenizer = ov.convert_model(semantic_detokenizer, example_input=semantic_tokens)

In [22]:
ov_global_detokenizer = ov.convert_model(global_detokenizer, example_input=global_tokens)
ov_global_detokenizer, global_tokens.shape

/app/sparktts/modules/fsq/residual_fsq.py:137: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert Q_indices == self.num_quantizers and self.num_quantizers == 1, \


(<Model: 'Model10'
 inputs[
 <ConstOutput: names[global_tokens] shape[?,?,?] type: i32>
 ]
 outputs[
 <ConstOutput: names[] shape[?,1024] type: f32>
 ]>,
 torch.Size([1, 1, 32]))

In [23]:
ov_prenet = ov.convert_model(prenet, example_input=(z_q, d_vector))
ov_prenet, z_q.shape, d_vector.shape

(<Model: 'Model12'
 inputs[
 <ConstOutput: names[z_q] shape[?,?,?] type: f32>,
 <ConstOutput: names[d_vector] shape[?,?] type: f32>
 ]
 outputs[
 <ConstOutput: names[] shape[?,1024,1..] type: f32>
 ]>,
 torch.Size([1, 1024, 497]),
 torch.Size([1, 1024]))

In [24]:
ov_vocoder = ov.convert_model(vocoder, example_input=(prenet_out, d_vector))

/app/sparktts/modules/blocks/layers.py:65: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if pad > 0:


In [41]:
ov_detokenizer = ov.convert_model(detokenizer, example_input=(semantic_tokens, global_tokens))

/app/sparktts/modules/fsq/residual_fsq.py:137: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert Q_indices == self.num_quantizers and self.num_quantizers == 1, \
/app/sparktts/modules/blocks/layers.py:65: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if pad > 0:


In [42]:
# Compile the OpenVINO model
device_name = "CPU"
core = ov.Core()
ovc_mel_spectrogram = core.compile_model(ov_mel_spectrogram, device_name)
ovc_wav2vec = core.compile_model(ov_wav2vec, device_name)
ovc_semantic_tokenizer = core.compile_model(ov_semantic_tokenizer, device_name)
ovc_global_tokenizer = core.compile_model(ov_global_tokenizer, device_name)
ovc_semantic_detokenizer = core.compile_model(ov_semantic_detokenizer, device_name)
ovc_global_detokenizer = core.compile_model(ov_global_detokenizer, device_name)
ovc_prenet = core.compile_model(ov_prenet, device_name)
ovc_vocoder = core.compile_model(ov_vocoder, device_name)

ovc_detokenizer = core.compile_model(ov_detokenizer, device_name)

In [26]:
# inference with OpenVINO
ov_audio = librosa.load("./ref.wav", sr=16000)[0]
# make ov_audio is the same shape as audio
# pad or trim
if len(ov_audio) < len(audio):
    ov_audio = np.pad(ov_audio, (0, len(audio) - len(ov_audio)), mode='constant')
elif len(ov_audio) > len(audio):
    ov_audio = ov_audio[:len(audio)]
ov_audio.shape, audio.shape

((159253,), (159253,))

In [27]:
ov_mel_input = torch.tensor(ov_audio).unsqueeze(0).unsqueeze(0)
ov_mel = ovc_mel_spectrogram(ov_mel_input) 
ov_mel[0].shape, ov_mel[0].max(), ov_mel[0].min()


((1, 128, 499), np.float32(2.1127803), np.float32(0.0))

In [28]:
ov_feat_input = torch.tensor(ov_audio).unsqueeze(0)
ov_feat = ovc_wav2vec(ov_feat_input)
ov_feat[0].shape, ov_feat[0].max(), ov_feat[0].min()

((1, 497, 1024), np.float32(685.2983), np.float32(-263.87338))

In [29]:
ov_semantic_tokens = ovc_semantic_tokenizer(ov_feat)
ov_semantic_tokens[0].shape

(1, 497)

In [30]:
ov_global_tokens = ovc_global_tokenizer(ov_mel)
ov_global_tokens[0].shape

(1, 1, 32)

In [31]:
ov_z_q = ovc_semantic_detokenizer(ov_semantic_tokens)
ov_z_q[0].shape

(1, 1024, 497)

In [32]:
ov_d_vector = ovc_global_detokenizer(ov_global_tokens)
ov_d_vector[0].shape

(1, 1024)

In [33]:
ov_prenet_out = ovc_prenet((ov_z_q[0], ov_d_vector[0]))
ov_prenet_out[0].shape

ov_prenet_out[0].shape

(1, 1024, 497)

In [34]:
ov_wav = ovc_vocoder((ov_prenet_out[0], ov_d_vector[0]))
ov_wav[0].shape

(1, 1, 159040)

In [35]:
Audio(ov_audio, rate=sr)  # Play the audio to verify it loaded correctly

In [36]:
ov_cpu_wav = ov_wav[0].squeeze().squeeze()
Audio(ov_cpu_wav, rate=sr)  # Play the audio to verify it

In [43]:
ov_wav2 = ovc_detokenizer((ov_semantic_tokens[0], ov_global_tokens[0]))
ov_wav2[0].shape, ov_wav2[0].max(), ov_wav2[0].min()

((1, 1, 159040), np.float32(0.08283389), np.float32(-0.0659463))

In [44]:
ov_cpu_wav2 = ov_wav2[0].squeeze().squeeze()
Audio(ov_cpu_wav2, rate=sr)  # Play the detokenized